In [ ]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from utils.utils import download_files, read_dataframe, save_model

ImportError: cannot import name 'download_files' from 'utils' (/Users/lark/files/learning/mlops-sandbox/utils/__init__.py)

In [ ]:
files_path = [
    './data/yellow_tripdata_2023-01.parquet',
    './data/yellow_tripdata_2023-02.parquet'
    ]
urls = [
    'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet',
    'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet'
    ]

In [ ]:
download_files(files_path, urls)

In [ ]:
df_train = pd.read_parquet(files_path[0])

In [ ]:
# Q1
len(df_train.columns)

In [ ]:
# Q2
df_train['duration'] = df_train['tpep_dropoff_datetime'] - df_train['tpep_pickup_datetime']
df_train['duration'] = df_train['duration'].apply(lambda td: td.total_seconds() / 60)
df_train.duration.std()

In [ ]:
# Q3
df_filtered = df_train[(df_train.duration >= 1) & (df_train.duration <= 60)].copy()
(df_filtered.shape[0] / df_train.shape[0]) * 100

In [ ]:
# Q4
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

# Convert categorical columns to string properly
for col in categorical:
    df_filtered[col] = df_filtered[col].fillna('').astype(str)

train_dicts = df_filtered[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

TARGET = 'duration'
y_train = df_filtered[TARGET].values

X_train.shape[1]


In [ ]:
# Q5
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)

root_mean_squared_error(y_train, y_pred)

In [ ]:
# Q6
df_val = pd.read_parquet(files_path[1])
df_val['duration'] = df_val.tpep_dropoff_datetime - df_val.tpep_pickup_datetime
df_val.duration = df_val.duration.apply(lambda td: td.total_seconds() / 60)
df_val_filtered = df_val[(df_val.duration >= 1) & (df_val.duration <= 60)].copy()
for col in categorical:
    df_val_filtered[col] = df_val_filtered[col].fillna('').astype(str)
val_dicts = df_val_filtered[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)
y_val = df_val_filtered[TARGET].values
y_pred = lr.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
rmse

In [ ]:
# Q6 correct validaion:)
# without filtering the outliers
df_val = pd.read_parquet(files_path[1])
df_val['duration'] = df_val.tpep_dropoff_datetime - df_val.tpep_pickup_datetime
df_val.duration = df_val.duration.apply(lambda td: td.total_seconds() / 60)
for col in categorical:
    df_val[col] = df_val[col].fillna('').astype(str)
val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)
y_val = df_val[TARGET].values
y_pred = lr.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
rmse